# NPPES National Dataset Creator (V2)

**What changed from v1:** The per-state filtering function now collects matching chunks into a Python list and calls `pd.concat` exactly once at the end, instead of calling `pd.concat([filtered_df, filtered_chunk])` inside the chunk loop.

Before saving the initial data files in the States folder, we then clean the data by...
- Mapping Specialties to Taxonomy
- Cleaning City column
- ZipCode to 5 digits
- Adding County

**Why it matters:** The v1 pattern is O(n²) — every concat copies the entire growing DataFrame in memory, so the work-per-chunk grows linearly with how many chunks have already been processed. Collecting frames in a list and concatenating once is O(n) and noticeably faster on the 11GB national file. Output is cleaner with extra data cleaning.

In [1]:
# Importing packages
import pandas as pd
import numpy as np

# Setting relevant file paths
dataDict_path = "/Users/lukebincarousky/Downloads/NPPES/NPPES/Dictionaries/main_nppes_data_dict.csv"
fullNPPES_path = "/Users/lukebincarousky/Downloads/NPPES/NPPES_Data_Dissemination_April_2026_V2/npidata_pfile_20050523-20260412.csv"

# Importing geographic workbook
geo = pd.read_excel("/Users/lukebincarousky/Downloads/NPPES/NPPES/Geographic Data/ZIP_Locale_Detail.xls")

# Saving the list of states as a separate variable
state_list = geo['PHYSICAL STATE'].unique()

# Selecting Relevant Columns

In [2]:
# Importing our data dictionary
data_dict = pd.read_csv(dataDict_path)  # type: ignore

# Creating a list of the columns we need to import using the data dictionary
cols = data_dict['Column_Name'].values

# Function to Read, Filter, Save Chunks of Dataset

The fix is in the body of `importStateChunk`: replace the in-loop `pd.concat` with list-append, then concat once. Data cleaning is also included in this block.

In [ ]:
def importStateChunk(state, fullNPPES_path, cols):
    # Establishing the results file path
    results_path = f"/Users/lukebincarousky/Downloads/NPPES/NPPES_Data_Dissemination_April_2026_V2/V2 States/{state} NPPES Extract.csv"

    # Collect matching chunks into a list, then concat once at the end (O(n) instead of O(n^2)).
    chunk_list = []
    chunksize = 100000
    for each_chunk in pd.read_csv(fullNPPES_path, chunksize=chunksize, usecols=cols, low_memory=False):
        filtered_chunk = each_chunk[each_chunk['Provider Business Practice Location Address State Name'] == state]
        if not filtered_chunk.empty:
            chunk_list.append(filtered_chunk)

    if chunk_list:
        filtered_df = pd.concat(chunk_list, ignore_index=True)
    else:
        # No rows for this state — produce an empty frame with the expected columns
        filtered_df = pd.DataFrame(columns=cols)

    # Dropping columns with all null values
    clean_df = filtered_df.dropna(axis=1, how='all')
    clean_df = clean_df.reset_index(drop=True)

    # Cleaning specified columns
    # ZipCodes to 5 digits
    clean_df['Provider Business Mailing Address Postal Code'] = clean_df['Provider Business Mailing Address Postal Code'].astype(str).str[:5]
    clean_df['Provider Business Practice Location Address Postal Code'] = clean_df['Provider Business Practice Location Address Postal Code'].astype(str).str[:5]

    # Phone numbers to 10 digits
    clean_df['Provider Business Mailing Address Telephone Number'] = clean_df['Provider Business Mailing Address Telephone Number'].astype(str).str[:10]
    clean_df['Provider Business Mailing Address Fax Number'] = clean_df['Provider Business Mailing Address Fax Number'].astype(str).str[:10]
    clean_df['Provider Business Practice Location Address Telephone Number'] = clean_df['Provider Business Practice Location Address Telephone Number'].astype(str).str[:10]
    clean_df['Provider Business Practice Location Address Fax Number'] = clean_df['Provider Business Practice Location Address Fax Number'].astype(str).str[:10]
    clean_df['Authorized Official Telephone Number'] = clean_df['Authorized Official Telephone Number'].astype(str).str[:10]

    # Mapping County to dataframe....

    # Mapping Specialty via Taxonomy code....

    # Saving the cleaned DataFrame to a new CSV file
    clean_df.to_csv(results_path, index=False)

    msg = f"Files Saved for {state}"
    return msg

# Looping Through States

In [ ]:
# List of items to remove
items_to_remove = ['PR', 'VI', np.nan, 'AS', 'GU', 'PW', 'FM', 'MP', 'MH']

# Removing nan & territories
state_list = [state for state in state_list if state not in items_to_remove]

In [5]:
# The list below loops through each state
for each_state in state_list:
    # The line below invokes our function
    result = importStateChunk(each_state, fullNPPES_path, cols)
    print(result)

Files Saved for MA
Files Saved for RI
Files Saved for NH
Files Saved for ME
Files Saved for VT
Files Saved for CT
Files Saved for NY
Files Saved for NJ
Files Saved for PA
Files Saved for DE
Files Saved for MD
Files Saved for DC
Files Saved for VA
Files Saved for WV
Files Saved for NC
Files Saved for SC
Files Saved for GA
Files Saved for TN
Files Saved for FL
Files Saved for AL
Files Saved for KY
Files Saved for MS
Files Saved for OH
Files Saved for IN
Files Saved for MI
Files Saved for IA
Files Saved for NE
Files Saved for WI
Files Saved for MN
Files Saved for ND
Files Saved for SD
Files Saved for MT
Files Saved for IL
Files Saved for MO
Files Saved for KS
Files Saved for LA
Files Saved for AR
Files Saved for TX
Files Saved for OK
Files Saved for NM
Files Saved for CO
Files Saved for WY
Files Saved for ID
Files Saved for WA
Files Saved for UT
Files Saved for AZ
Files Saved for NV
Files Saved for CA
Files Saved for HI
Files Saved for OR
Files Saved for AK
